In [2]:
import scanpy as sc
from anndata.experimental import concat_on_disk
from umap import UMAP
import numpy as np

import bionty as bt
from scdataloader.utils import get_all_ancestors
from pybiomart import Dataset

import matplotlib
import datamapplot
import json
import os
import ssl
import urllib3
from huggingface_hub import hf_hub_download

# Additional imports for scPRINT
from scprint import scPrint
from scdataloader import Preprocessor
from scdataloader.preprocess import additional_preprocess
from scdataloader.utils import load_genes
import torch
import pandas as pd

→ connected lamindb: anonymous/test
No module named 'triton'
FlashAttention is not installed, not using it..


# Prepare object

In [ ]:
adata = sc.read_h5ad("data/gut_data/gut_hs_full_annotated_AM_06032025_140458_raw.h5ad")

In [ ]:
with open('ontology_terms.json', 'r') as f:
    ontology_data = json.load(f)

cell_type_mapping = {item['cell_annotation']: item['ontology_term'] 
                     for item in ontology_data['ontology_terms']}
tissue_mapping = {item['tissue']: item['ontology_term'] 
                  for item in ontology_data['tissue_mapping']}
sex_mapping = ontology_data['sex_mapping']
age_group_mapping = ontology_data['age_group_mapping']
assay_mapping = ontology_data['assay_mapping']

In [ ]:
adata.obs['organism'] = 'Homo sapiens'
adata.obs['organism_ontology_term_id'] = 'NCBITaxon:9606'

In [ ]:
# Map cell_states to cell_type_ontology_term_id
adata.obs['cell_type_ontology_term_id'] = 'unknown'
for cell_type in adata.obs['cell_states'].unique():
    if cell_type in cell_type_mapping:
        ontology_id = cell_type_mapping[cell_type]
        mask = adata.obs['cell_states'] == cell_type
        adata.obs.loc[mask, 'cell_type_ontology_term_id'] = ontology_id
    else:
        print(f"No mapping found for '{cell_type}', keeping as 'unknown'")

In [ ]:
# Map age_group to development_stage_ontology_term_id
adata.obs['development_stage_ontology_term_id'] = adata.obs['age_group'].map(age_group_mapping)

# Map gut_region to tissue_ontology_term_id 
if 'gut_region' in adata.obs.columns:
    adata.obs['tissue_ontology_term_id'] = adata.obs['gut_region'].map(tissue_mapping)
else:
    adata.obs['tissue_ontology_term_id'] = 'unknown'

In [ ]:
# Map library_preparation_protocol to assay_ontology_term_id
protocol_str = adata.obs['library_preparation_protocol'].astype(str)
adata.obs['assay_ontology_term_id'] = protocol_str.map(assay_mapping).fillna('EFO:0008913')

In [ ]:
adata.obs['self_reported_ethnicity_ontology_term_id'] = 'unknown'
adata.obs['disease_ontology_term_id'] = 'PATO:0000461'  # normal

In [ ]:
# Map sex to sex_ontology_term_id 
if 'sex' in adata.obs.columns:
    # Convert to string to handle categorical issues
    adata.obs['sex_normalized'] = adata.obs['sex'].astype(str).str.lower()
    adata.obs['sex_ontology_term_id'] = adata.obs['sex_normalized'].map(sex_mapping).fillna('unknown')
    adata.obs.drop('sex_normalized', axis=1, inplace=True)
else:
    adata.obs['sex_ontology_term_id'] = 'unknown'

In [ ]:
required_cols = ['organism_ontology_term_id', 'cell_type_ontology_term_id', 'tissue_ontology_term_id', 
                'assay_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id',
                'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id']
missing_cols = [col for col in required_cols if col not in adata.obs.columns]
print(f"WARNING: Missing required columns: {missing_cols}")

In [ ]:
adata.var["gene_symbol"] = adata.var.index.copy()

In [ ]:
dataset = Dataset(name='hsapiens_gene_ensembl', host='http://www.ensembl.org')

In [ ]:
biomart_results = dataset.query(attributes=['ensembl_gene_id', 'external_gene_name'])

In [ ]:
# Create a mapping dictionary from biomart_results
gene_id_map = biomart_results.set_index('Gene name')['Gene stable ID'].to_dict()

# Add ensembl gene IDs to adata.var
adata.var['gene_id-query'] = adata.var['gene_symbol'].map(gene_id_map)

# Fill missing values with the corresponding index values
adata.var['gene_id-query'] = adata.var['gene_id-query'].fillna(adata.var.index.to_series())

In [ ]:
adata.var.index = adata.var['gene_id-query'].copy()

In [ ]:
# Check for duplicates in var index
duplicates = adata.var.index.duplicated()
duplicates

array([False, False, False, ..., False, False, False])

In [ ]:
if adata.var.index.duplicated().any():
    adata = adata[:, ~adata.var.index.duplicated(keep='first')]

### Loading the model

In [ ]:
# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Set environment variables to disable SSL verification
os.environ['PYTHONHTTPSVERIFY'] = '0'
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['SSL_VERIFY'] = 'false'

# Configure SSL context
ssl._create_default_https_context = ssl._create_unverified_context

# Configure requests to not verify SSL
import requests.packages.urllib3.util.ssl_
requests.packages.urllib3.util.ssl_.DEFAULT_CIPHERS = 'ALL'

# Monkey patch requests to disable SSL verification
original_request = requests.Session.request
def patched_request(self, method, url, **kwargs):
    kwargs['verify'] = False
    return original_request(self, method, url, **kwargs)
requests.Session.request = patched_request

# Now download the model
model_checkpoint_file = hf_hub_download(
    repo_id="jkobject/scPRINT", 
    filename="v2-medium.ckpt"
)

In [ ]:
model_checkpoint_file

'/Users/am336941/.cache/huggingface/hub/models--jkobject--scPRINT/snapshots/d79fad0ff77e19d9ba0c2fad9dfc993b37e92d6d/v2-medium.ckpt'

In [ ]:
# Set up preprocessing for scPRINT
torch.set_float32_matmul_precision('medium')

# Create preprocessor for scPRINT
preprocessor = Preprocessor(
    do_postp=True,
    additional_preprocess=additional_preprocess,
    force_preprocess=True,
)

# Ensure gene IDs are in the right format
adata.var.index = adata.var["gene_id-query"].astype(str).copy()

# Preprocess the data for scPRINT
adata_processed = preprocessor(adata)

Dropping layers:  KeysView(Layers with keys: )
checking raw counts
removed 0 non primary cells, 365542 renamining
removed 0 non primary cells, 365542 renamining
filtered out 1515 cells, 364027 renamining
filtered out 1515 cells, 364027 renamining
Removed 1403 genes not known to the ontology
Removed 1403 genes not known to the ontology
Removed 0 duplicate genes
Added 28325 genes in the ontology but not present in the dataset
Removed 0 duplicate genes
Added 28325 genes in the ontology but not present in the dataset
validating
validating


/Users/am336941/uv_envs/scprint_env230/scprint_env230/lib/python3.10/site-packages/scdataloader/preprocess.py:287: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  adata, organism=adata.obs.organism_ontology_term_id[0], need_all=False


! received 5 unique terms, 364022 empty/duplicated terms are ignored
! 1 unique term (20.00%) is not validated for ontology_id: 'unknown'
! 1 unique term (20.00%) is not validated for ontology_id: 'unknown'
starting QC
starting QC
Seeing 123607 outliers (33.96% of total dataset):
normalize
Seeing 123607 outliers (33.96% of total dataset):
normalize
starting PCA
starting PCA
done
AnnData object with n_obs × n_vars = 364027 × 70611
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_coun

In [ ]:
# Load the model properly
try:
    m = torch.load(model_checkpoint_file)
except RuntimeError: 
    m = torch.load(model_checkpoint_file, map_location=torch.device('cpu'))

# Set transformer type based on GPU availability
transformer = "flash" if torch.cuda.is_available() else "normal"

# Handle compatibility issues with different model versions
if "prenorm" in m['hyper_parameters']:
    m['hyper_parameters'].pop("prenorm")
    torch.save(m, model_checkpoint_file)

if "label_counts" in m['hyper_parameters']:
    model = scPrint.load_from_checkpoint(
        model_checkpoint_file, 
        precpt_gene_emb=None, 
        classes=m['hyper_parameters']['label_counts'], 
        transformer=transformer
    )
else:
    model = scPrint.load_from_checkpoint(
        model_checkpoint_file, 
        precpt_gene_emb=None, 
        transformer=transformer
    )

RuntimeError caught: scPrint is not attached to a `Trainer`.


In [ ]:
# Handle gene compatibility issues
missing = set(model.genes) - set(load_genes(model.organisms).index)
if len(missing) > 0:
    print(f"Warning: {len(missing)} genes mismatch between model and ontology")

# Configure model device and precision
if not torch.cuda.is_available():
    model = model.to(torch.float32)
    
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded successfully on {device} with {dtype} precision")

Model loaded successfully on cpu with torch.float32 precision


# Generate nice umap

In [ ]:
os.makedirs("./data", exist_ok=True)
adata_processed.write_h5ad("./data/preprocessed_for_prediction.h5ad")

... storing 'organism' as categorical
... storing 'organism_ontology_term_id' as categorical
... storing 'organism_ontology_term_id' as categorical
... storing 'cell_type_ontology_term_id' as categorical
... storing 'assay_ontology_term_id' as categorical
... storing 'cell_type_ontology_term_id' as categorical
... storing 'assay_ontology_term_id' as categorical
... storing 'self_reported_ethnicity_ontology_term_id' as categorical
... storing 'disease_ontology_term_id' as categorical
... storing 'self_reported_ethnicity_ontology_term_id' as categorical
... storing 'disease_ontology_term_id' as categorical
... storing 'sex_ontology_term_id' as categorical
... storing 'cell_culture' as categorical
... storing 'sex_ontology_term_id' as categorical
... storing 'cell_culture' as categorical
... storing 'batches' as categorical
... storing 'batches' as categorical
... storing 'gene_symbol' as categorical
... storing 'gene_symbol' as categorical
... storing 'gene_id-query' as categorical
... s

In [4]:
adata

AnnData object with n_obs × n_vars = 365542 × 43704
    obs: 'cell_index', 'Source Name', 'ENA_SAMPLE', 'BioSD_SAMPLE', 'organism', 'disease', 'organism_part', 'cell_type', 'growth_condition', 'developmental_stage', 'Material Type', 'Protocol REF', 'sample_id', 'LIBRARY_LAYOUT', 'cdna_read_size', 'cell_barcode_size', 'end_bias', 'library_construction', 'sample_barcode_size', 'umi_barcode_offset', 'umi_barcode_size', 'Performer', 'Assay Name', 'ENA_EXPERIMENT', 'ENA_RUN', 'time', 'time_unit', 'n_genes', 'doublet_scores', 'predicted_doublets', 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'percent_mito', 'n_counts_mito', 'percent_ribo', 'n_counts_ribo', 'percent_hb', 'n_counts_hb', 'percent_top50', 'cell_passed_qc', 'qc_cluster', 'cluster_passed_qc', 'consensus_fraction', 'consensus_passed_qc', 'total_counts', 'n_genes_by_counts', 'percent_chrY', 'XIST-counts', 'XIST-percentage', 'sex', 'S_score', 'G2M_score', 'Cell_cycle_phase', 'Study_name', 'ArrayExpress_ID', 'library_preparation_pro

In [8]:
adata = sc.read_h5ad("/Users/am336941/PhD/data/gut_data/gut_hs_all_datasets_full_annotated_AM_03032025_141544_raw.h5ad")

In [11]:
adata.obs

,cell_index,Source Name,ENA_SAMPLE,BioSD_SAMPLE,organism,disease,organism_part,cell_type,growth_condition,developmental_stage,Material Type,Protocol REF,sample_id,LIBRARY_LAYOUT,cdna_read_size,cell_barcode_size,end_bias,library_construction,sample_barcode_size,umi_barcode_offset,umi_barcode_size,Performer,Assay Name,ENA_EXPERIMENT,ENA_RUN,time,time_unit,n_genes,doublet_scores,predicted_doublets,n_counts,log1p_n_counts,log1p_n_genes,percent_mito,n_counts_mito,percent_ribo,n_counts_ribo,percent_hb,n_counts_hb,percent_top50,cell_passed_qc,qc_cluster,cluster_passed_qc,consensus_fraction,consensus_passed_qc,total_counts,n_genes_by_counts,percent_chrY,XIST-counts,XIST-percentage,sex,S_score,G2M_score,Cell_cycle_phase,Study_name,ArrayExpress_ID,library_preparation_protocol,full_age,age_group,immunophenotype,metadata_cluster,barcode,category,Integrated_05,age,gestational_age,donor_id,passage,batch,sampling_site,celltype,library_construnction_and_layout,cell_states,cellstates_scANVI,confidence_score,gut_region,cell_state
AAAGAACCACTGAATC-1-HT071-LWRN-EGF_Epithelial,AAAGAACCACTGAATC-1-HT071-LWRN-EGF,HT071-LWRN-EGF,ERS5297462,SAMEA7541009,Homo sapiens,normal,duodenum,intestinal epithelial cell,"fetal enteroid grown medium EGF, 100 nanogram ...",late embryonic stage,cell,P-MTAB-102913,HT071-LWRN-EGF,PAIRED,91,16,3 prime tag,10xV3,8,16,12,University of Michigan Advanced Genomics Core,50pct-LWRN-E_S88_L000,ERX4678371,ERR4808810,142.0,day,2665,0.197903,False,10613,9.269929,7.888335,15.141807,1607.0,22.679732,2407.0,0.0,0.0,37.802695,True,4,True,1.0,True,10613,2665,0.084802,0,0.000000,male,-0.186563,-0.578972,G1,Holloway_2021,E-MTAB-9720,10xV3 3 prime tag,20.3 week,cell culture model,unsorted,0,AAAGAACCACTGAATC,Unknown,Unknown,nan,N/A - not fetal,cell_culture,N/A,unknown,unknown,Epithelial,10xV3_PAIRED,Stem cells,Stem cells,0.838225,small intestine,NaN
AAATGGACACCCTAAA-1-HT071-LWRN-EGF_Epithelial,AAATGGACACCCTAAA-1-HT071-LWRN-EGF,HT071-LWRN-EGF,ERS5297462,SAMEA7541009,Homo sapiens,normal,duodenum,intestinal epithelial cell,"fetal enteroid grown medium EGF, 100 nanogram ...",late embryonic stage,cell,P-MTAB-102913,HT071-LWRN-EGF,PAIRED,91,16,3 prime tag,10xV3,8,16,12,University of Michigan Advanced Genomics Core,50pct-LWRN-E_S88_L000,ERX4678371,ERR4808810,142.0,day,3318,0.047002,False,14015,9.547955,8.107419,15.155191,2124.0,20.920442,2932.0,0.0,0.0,34.541563,True,1,True,1.0,True,14015,3318,0.071352,0,0.000000,male,1.911555,1.461709,S,Holloway_2021,E-MTAB-9720,10xV3 3 prime tag,20.3 week,cell culture model,unsorted,0,AAATGGACACCCTAAA,Unknown,Unknown,nan,N/A - not fetal,cell_culture,N/A,unknown,unknown,Epithelial,10xV3_PAIRED,TA,TA,0.999985,small intestine,NaN
AACCACAGTGATACTC-1-HT071-LWRN-EGF_Epithelial,AACCACAGTGATACTC-1-HT071-LWRN-EGF,HT071-LWRN-EGF,ERS5297462,SAMEA7541009,Homo sapiens,normal,duodenum,intestinal epithelial cell,"fetal enteroid grown medium EGF, 100 nanogram ...",late embryonic stage,cell,P-MTAB-102913,HT071-LWRN-EGF,PAIRED,91,16,3 prime tag,10xV3,8,16,12,University of Michigan Advanced Genomics Core,50pct-LWRN-E_S88_L000,ERX4678371,ERR4808810,142.0,day,3551,0.369803,False,17921,9.793784,8.175266,18.135149,3250.0,23.966297,4295.0,0.0,0.0,39.925227,True,4,True,1.0,True,17921,3551,0.100441,0,0.000000,male,-0.739878,-0.556376,G1,Holloway_2021,E-MTAB-9720,10xV3 3 prime tag,20.3 week,cell culture model,unsorted,0,AACCACAGTGATACTC,Unknown,Unknown,nan,N/A - not fetal,cell_culture,N/A,unknown,unknown,Epithelial,10xV3_PAIRED,EECs,EEC,0.992497,small intestine,NaN
AACCATGTCACCCTTG-1-HT071-LWRN-EGF_Epithelial,AACCATGTCACCCTTG-1-HT071-LWRN-EGF,HT071-LWRN-EGF,ERS5297462,SAMEA7541009,Homo sapiens,normal,duodenum,intestinal epithelial cell,"fetal enteroid grown medium EGF, 100 nanogram ...",late embryonic stage,cell,P-MTAB-102913,HT071-LWRN-EGF,PAIRED,91,16,3 prime tag,10xV3,8,16,12,University of Michigan Advanced Genomics Core,50pct-LWRN-E_S88_L000,ERX4678371,ERR4808810,142.0,day,3749,0.340122,False,19435,9.874882,8.229511,19.902238,3868.0,